In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd

from shapely.geometry import LineString, MultiLineString
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from shapely.geometry import box
import matplotlib.colors as mcolors

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# ============================
# User settings
# ============================
sdate, edate = '20090701', '20240630'
write_path = '/scratch/ng72/ms5578/solar_wind_tseries/'

# BARRA-R2 variable paths
data = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ta100m/latest/"

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")
cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")

# Dask cluster

client = Client(n_workers=12,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36501 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/36501/status,
Dashboard: /proxy/36501/status,Workers: 12
Total threads: 12,Total memory: 55.88 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39931,Workers: 0
Dashboard: /proxy/36501/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34491,Total threads: 1
Dashboard: /proxy/45923/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:34733,


In [2]:
boco = cluster_dates[cluster_dates['BOCORWF1'] == 1]

In [3]:
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GULLRWF2',
            'GUNNING1',
            'BANGOWF1',
            'BANGOWF2',
            'COLWF01',
            'WOODLWN1',
            'BOCORWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [4]:
def get_days(days, nc_dir):
    extent = [147.5, 151, -38.5, -33.5]
    lon_min, lon_max, lat_min, lat_max = extent
    step=2
        
    date_list = pd.to_datetime(days['date'])

    # Build filename filter
    all_files = os.listdir(nc_dir)
    selected_files = [
        os.path.join(nc_dir, f)
        for f in all_files
        if any(d.strftime("%Y%m") in f for d in date_list)
    ]

    # Open multiple files lazily with parallel reads
    ds = xr.open_mfdataset(
        selected_files,
        combine='by_coords',
        parallel=True,
        chunks='auto'
    )
    
    # Select all hours of the requested dates
    subset = ds.where(ds.time.dt.floor('D').isin(date_list), drop=True)

    subset = subset.sel(**{
    'lon': slice(lon_min, lon_max),
    'lat': slice(lat_min, lat_max)
    })

    # Convert to Australia/Sydney
    local_time = (
        pd.DatetimeIndex(subset.time.values)
        .tz_localize("UTC")
        .tz_convert("Australia/Sydney")
    )
    
    # Drop tzinfo so xarray can store it
    local_time_naive = local_time.tz_localize(None)
    
    # Assign back to dataset
    subset = subset.assign_coords(time=local_time_naive)

    return subset


In [5]:
t100_boco = get_days(boco, data)

t100_boco = t100_boco.chunk({"time": 168, "lat": -1, "lon": -1})

In [6]:
# Lazy hourly mean
hourly_composite =  t100_boco.groupby("time.hour").mean()

# Compute in parallel
with ProgressBar():
    hourly_composite = hourly_composite.compute()
    hourly_composite = hourly_composite.assign_coords(hour=("hour", np.arange(24)))

In [7]:
def plot_divergence_frame(divergence, lat, lon, t, cluster=cluster, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite',
                          extent=[147.5, 151, -38.5, -33.5]):
    """
    Plot scalar divergence with optional cluster points and contour shapefile.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)

    norm = mcolors.Normalize(vmin=280, vmax=320)
    
    cf = ax.contourf(lon, lat, divergence, cmap='plasma', norm=norm,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label='Temperature (K)')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'Composite of 100m air temperature at hour {str(t)} (AEST)')
    
    # Save
    filename = f"{output_dir}/ta100m_hour_{t}.png"
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(filename)
    return filename

In [8]:
# Load shapefile
gdf = gpd.read_file('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/contours/aus25cgd_l.shp').to_crs(epsg=4326)
bbox = box(147.5, -38.5, 151, -33.5)
gdf_clip = gdf[gdf.geometry.intersects(bbox)]

In [10]:
for t in range(24):
    lat_t = hourly_composite['lat'].values
    lon_t = hourly_composite['lon'].values

    values = hourly_composite['ta100m'].isel(hour=t).values
    
    plot_divergence_frame(values, lat_t, lon_t, t, cluster=cluster, highlight_id='BOCORWF1',
                              shapefile_gdf=gdf,
                              output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite',
                              extent=[147.5, 151, -38.5, -33.5])

/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_0.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_1.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_2.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_3.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_4.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_5.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_6.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_7.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_8.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_9.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/ta100m_composite/ta100m_hour_10.pn